# Pythia longitudinal CPS sweep

Coupling-Phase Spectroscopy — governed Colab runner.

Runs a configurable checkpoint sequence. The default grid is deliberately small; set `CPS_REVISIONS` to a comma-separated list for the complete campaign.

In [ ]:
import os, pathlib, subprocess, sys
REPO_URL = os.environ.get("CPS_REPO_URL", "https://github.com/fyremael/CPS.git")
GIT_REF = os.environ.get("CPS_GIT_REF", "main")
repo = pathlib.Path("/content/CPS")
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", GIT_REF, REPO_URL, str(repo)], check=True)
else:
    subprocess.run(["git", "-C", str(repo), "fetch", "origin", GIT_REF], check=True)
    subprocess.run(["git", "-C", str(repo), "checkout", GIT_REF], check=True)
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
os.chdir(repo)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[pythia,notebooks]"], check=True)
print("repo", repo, "ref", GIT_REF)

In [ ]:
import os
from cps.pythia.config import load_probe_config
from cps.pythia.runner import run_longitudinal
revisions = os.environ.get("CPS_REVISIONS", "step0,step1,step16,step512,step1000").split(",")
config = load_probe_config("subjects/pythia/configs/pythia_70m_longitudinal.yaml")
summary = run_longitudinal(config, revisions)
print(summary)

In [ ]:
import pathlib, shutil
export_dir = pathlib.Path("/content/cps-export")
export_dir.mkdir(parents=True, exist_ok=True)
source = pathlib.Path("/content/cps-artifacts")
if source.exists():
    shutil.copytree(source, export_dir / "artifacts", dirs_exist_ok=True)
shutil.make_archive("/content/cps-export", "zip", "/content/cps-export")
print("exported", export_dir)